<a href="https://colab.research.google.com/github/ibtisamelahi07/generativeai/blob/main/Task4_Speech_to_Reasoning_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ Task 4: Speech-to-Reasoning Pipeline — Whisper + Quantized LLM

**Pipeline Overview:**
1. **ASR** — OpenAI Whisper (`whisper-small`) transcribes audio to text via HuggingFace pipeline
2. **Reasoning LLM** — Unsloth dynamic 4-bit Qwen2.5-3B-Instruct generates step-by-step reasoning
3. **Batching** — `batch_size=8` for Whisper chunks; sequential LLM queries with VRAM tracking
4. **Memory** — fp16 Whisper + dynamic 4-bit LLM keeps total usage under 3 GB

**Runtime:** GPU T4 (16 GB VRAM)  
**ASR:** `openai/whisper-small` (fp16)  
**Reasoning LLM:** `unsloth/Qwen2.5-3B-Instruct-bnb-4bit`

---
## 🔧 Section 1: Install Dependencies

In [ ]:
%%capture
# ── Unsloth dynamic 4-bit quantization ──────────────────────────
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

# ── Whisper via HuggingFace Transformers (supports batching) ────
!pip install transformers datasets

# ── Audio processing ─────────────────────────────────────────────
!pip install torchaudio librosa soundfile

# ── Sample audio generation ──────────────────────────────────────
!pip install gTTS

# ── Utilities ────────────────────────────────────────────────────
!pip install numpy pandas

# ffmpeg: required by Whisper for mp3/m4a decoding
!apt-get install -y ffmpeg > /dev/null 2>&1

print("All dependencies installed.")

---
## 🖥️ Section 2: GPU & VRAM Utilities

In [ ]:
import torch
import gc
import time
import os

def print_vram(label=""):
    """Report VRAM status at any checkpoint."""
    if not torch.cuda.is_available():
        print("No GPU detected.")
        return
    alloc = torch.cuda.memory_allocated() / 1e9
    resrv = torch.cuda.memory_reserved()  / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    tag   = f"[{label}] " if label else ""
    print(f"{tag}VRAM -> Allocated: {alloc:.2f} GB | Reserved: {resrv:.2f} GB | Free: {total-resrv:.2f} GB / {total:.2f} GB total")

def free_vram():
    """Release unused GPU memory."""
    gc.collect()
    torch.cuda.empty_cache()

if not torch.cuda.is_available():
    raise RuntimeError("GPU required. Go to Runtime > Change runtime type > T4 GPU.")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA: {torch.version.cuda} | PyTorch: {torch.__version__}")
print()
print_vram("Baseline")

GPU: Tesla T4
CUDA: 12.8 | PyTorch: 2.11.0+cu128

[Baseline] VRAM -> Allocated: 0.00 GB | Reserved: 0.00 GB | Free: 15.64 GB / 15.64 GB total


---
## 🎤 Section 3: Load Whisper ASR Model

**Configuration choices for efficiency:**
- `torch_dtype=float16` halves VRAM vs fp32 (~500 MB on GPU)
- `chunk_length_s=30` enables sliding-window processing of long audio
- `stride_length_s=(4,2)` overlapping stride prevents boundary word cuts
- `batch_size=8` processes 8 audio chunks in parallel on the T4

In [ ]:
from transformers import pipeline as hf_pipeline

# ── Whisper config ────────────────────────────────────────────────
WHISPER_MODEL  = "openai/whisper-small"   # 244M params, ~500 MB VRAM in fp16
ASR_BATCH_SIZE = 8                         # parallel audio chunks
CHUNK_LEN_S   = 30                        # sliding window length (seconds)
STRIDE_LEN     = (4, 2)                   # (left_overlap, right_overlap) seconds

print(f"Loading Whisper ASR: {WHISPER_MODEL}")
print("Running fp16 on GPU, batch_size=8 for efficient chunked processing...\n")

asr_pipe = hf_pipeline(
    task            = "automatic-speech-recognition",
    model           = WHISPER_MODEL,
    device          = 0,                   # GPU
    torch_dtype     = torch.float16,       # fp16: ~500 MB VRAM
    chunk_length_s  = CHUNK_LEN_S,
    stride_length_s = STRIDE_LEN,
)

print("Whisper ASR pipeline ready.")
print_vram("After Whisper load")

Loading Whisper ASR: openai/whisper-small
Running fp16 on GPU, batch_size=8 for efficient chunked processing...



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Whisper ASR pipeline ready.
[After Whisper load] VRAM -> Allocated: 0.50 GB | Reserved: 0.52 GB | Free: 15.11 GB / 15.64 GB total


---
## 🤖 Section 4: Load Unsloth Dynamic 4-bit Reasoning LLM

**Why Qwen2.5-3B-Instruct?**
- State-of-the-art reasoning on math, logic, and Q&A tasks
- Dynamic 4-bit: attention layers preserved at higher precision for better reasoning
- ~2 GB VRAM — leaves 13+ GB free alongside Whisper on the T4

In [ ]:
from unsloth import FastLanguageModel

LLM_MODEL      = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
MAX_SEQ_LEN    = 2048

print(f"Loading reasoning LLM: {LLM_MODEL}")
print("Dynamic 4-bit quantization — critical attention weights preserved at higher precision...\n")

llm_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = LLM_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,       # auto: float16 on T4
    load_in_4bit   = True,       # Unsloth dynamic 4-bit
)

# Activate Unsloth's inference mode (Flash Attention 2 + no gradient tracking)
FastLanguageModel.for_inference(llm_model)

print("\nReasoning LLM loaded in inference mode.")
print_vram("After LLM load")
print()
print("Both models loaded. Combined VRAM usage:")
print("  Whisper-small fp16 : ~0.5 GB")
print("  Qwen2.5-3B 4-bit   : ~2.0 GB")
print("  Total              : ~2.5 GB (well within T4 budget)")


/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:153: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading reasoning LLM: unsloth/Qwen2.5-3B-Instruct-bnb-4bit
Dynamic 4-bit quantization — critical attention weights preserved at higher precision...

==((====))==  Unsloth 2026.6.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-3B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.

Reasoning LLM loaded in inference mode.
[After LLM load] VRAM -> Allocated: 2.60 GB | Reserved: 2.62 GB | Free: 13.01 GB / 15.64 GB total

Both models loaded. Combined VRAM usage:
  Whisper-small fp16 : ~0.5 GB
  Qwen2.5-3B 4-bit   : ~2.0 GB
  Total              : ~2.5 GB (well within T4 budget)


---
## 🔊 Section 5: Generate Sample Audio Files

gTTS synthesises `.mp3` files from text — simulating real spoken questions across four reasoning categories.

In [ ]:
from gtts import gTTS
from IPython.display import Audio, display

# ── Four reasoning-type audio queries ────────────────────────────
AUDIO_QUERIES = [
    {
        "id"  : "audio_1",
        "type": "Mathematical Reasoning",
        "text": (
            "A train leaves Station A at 8 AM travelling at 90 kilometres per hour. "
            "Another train leaves Station B, which is 360 kilometres away, at 9 AM "
            "travelling at 60 kilometres per hour toward Station A. "
            "At what time will the two trains meet?"
        )
    },
    {
        "id"  : "audio_2",
        "type": "Logical Syllogism",
        "text": (
            "All engineers are problem solvers. "
            "Some problem solvers are creative thinkers. "
            "Can we conclude that some engineers are creative thinkers? "
            "Explain your reasoning step by step."
        )
    },
    {
        "id"  : "audio_3",
        "type": "Technical Question Answering",
        "text": (
            "What is the difference between supervised learning and unsupervised learning "
            "in machine learning? Give a practical example of each."
        )
    },
    {
        "id"  : "audio_4",
        "type": "Industrial Engineering Problem",
        "text": (
            "In a factory, Machine A produces 200 units per hour and Machine B produces "
            "150 units per hour. Machine A runs for 3 hours, then breaks down. "
            "Machine B continues for 2 more hours after that. "
            "What is the total production across both machines?"
        )
    },
]

# ── Generate audio files ──────────────────────────────────────────
audio_paths = []
for q in AUDIO_QUERIES:
    path = f"{q['id']}.mp3"
    gTTS(text=q["text"], lang="en", slow=False).save(path)
    audio_paths.append(path)
    size_kb = os.path.getsize(path) / 1024
    print(f"Generated: {path}  [{q['type']}]  ({size_kb:.1f} KB)")

print(f"\n{len(audio_paths)} audio files ready.")

# Preview in notebook
print("\nPreview Audio 1 (Mathematical Reasoning):")
display(Audio("audio_1.mp3", autoplay=False))

Generated: audio_1.mp3  [Mathematical Reasoning]  (150.2 KB)
Generated: audio_2.mp3  [Logical Syllogism]  (101.4 KB)
Generated: audio_3.mp3  [Technical Question Answering]  (69.9 KB)
Generated: audio_4.mp3  [Industrial Engineering Problem]  (157.1 KB)

4 audio files ready.

Preview Audio 1 (Mathematical Reasoning):


---
## ✍️ Section 6: Whisper Transcription Functions

In [ ]:
def transcribe_audio(audio_path: str, verbose: bool = True) -> str:
    """
    Transcribe one audio file with Whisper.

    Uses batch_size=8 for parallel chunk processing.
    Forces English language to skip auto-detection overhead.

    Args:
        audio_path : Path to .mp3 / .wav / .flac file.
        verbose    : Print result.
    Returns:
        Transcribed text string.
    """
    result = asr_pipe(
        audio_path,
        batch_size      = ASR_BATCH_SIZE,
        return_timestamps = False,
        generate_kwargs = {"language": "english", "task": "transcribe"},
    )
    text = result["text"].strip()
    if verbose:
        print(f"  Transcription: {text}")
    return text


# ── Single test ───────────────────────────────────────────────────
print("Testing Whisper on audio_1.mp3...")
_ = transcribe_audio("audio_1.mp3")
print_vram("After transcription test")

Testing Whisper on audio_1.mp3...


A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


  Transcription: a train leaves station A at 8 a.m. traveling at 90 kilometers per hour. Another train leaves station B, which is 360 kilometers away, at 9 a.m. traveling at 60 kilometers per hour toward station A. At what time will the two trains meet?
[After transcription test] VRAM -> Allocated: 2.60 GB | Reserved: 2.98 GB | Free: 12.65 GB / 15.64 GB total


---
## 🧠 Section 7: Reasoning Generation Functions

In [ ]:
SYSTEM_PROMPT = (
    "You are a precise step-by-step reasoning assistant. "
    "For each question, think through it methodically, "
    "show your working clearly, and state a concise final answer."
)


def build_prompt(text: str) -> str:
    """Format input as Qwen2.5 ChatML via apply_chat_template."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": text},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize              = False,
        add_generation_prompt = True,  # appends <|im_start|>assistant\n
    )


def generate_reasoning(
    text           : str,
    max_new_tokens : int   = 450,
    temperature    : float = 0.4,
    top_p          : float = 0.9,
    verbose        : bool  = True,
) -> str:
    """
    Generate a reasoned response via the quantized LLM.

    Args:
        text           : Transcribed question text.
        max_new_tokens : Max tokens to generate.
        temperature    : Sampling temperature.
        top_p          : Nucleus sampling cutoff.
        verbose        : Print token count.
    Returns:
        Reasoning string.
    """
    prompt = build_prompt(text)

    inputs = tokenizer(
        prompt,
        return_tensors = "pt",
        truncation     = True,
        max_length     = MAX_SEQ_LEN,
    ).to("cuda")

    if verbose:
        print(f"  Prompt tokens: {inputs['input_ids'].shape[1]}")

    with torch.no_grad():
        out_ids = llm_model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            do_sample          = True,
            temperature        = temperature,
            top_p              = top_p,
            repetition_penalty = 1.1,
            eos_token_id       = tokenizer.eos_token_id,
            pad_token_id       = tokenizer.eos_token_id,
        )

    new_ids  = out_ids[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    return response


print("Reasoning functions defined.")

Reasoning functions defined.


---
## 🔗 Section 8: End-to-End Pipeline Function

In [ ]:
def speech_to_reasoning(
    audio_path     : str,
    query_label    : str   = "",
    max_new_tokens : int   = 450,
    temperature    : float = 0.4,
    verbose        : bool  = True,
) -> dict:
    """
    Full Speech-to-Reasoning pipeline:
      Stage 1 — ASR  : Audio file -> Whisper -> Transcription
      Stage 2 — LLM  : Transcription -> Qwen2.5-3B-4bit -> Reasoned response

    Returns dict: audio_path, transcription, reasoning,
                  asr_time_s, llm_time_s, total_time_s, vram_gb
    """
    label = query_label or audio_path
    if verbose:
        print("=" * 65)
        print(f"QUERY: {label}")
        print("=" * 65)

    # Stage 1: ASR
    if verbose: print("\n[Stage 1 / ASR] Whisper Transcription...")
    t0 = time.time()
    transcription = transcribe_audio(audio_path, verbose=verbose)
    asr_time = round(time.time() - t0, 2)
    if verbose: print(f"  ASR time: {asr_time}s")

    # Stage 2: Reasoning
    if verbose: print("\n[Stage 2 / LLM] Qwen2.5 Reasoning...")
    t1 = time.time()
    reasoning = generate_reasoning(transcription, max_new_tokens=max_new_tokens,
                                    temperature=temperature, verbose=verbose)
    llm_time = round(time.time() - t1, 2)

    total_time = round(time.time() - t0, 2)
    vram_gb    = round(torch.cuda.memory_allocated() / 1e9, 2)

    if verbose:
        print(f"\n  Reasoned Response:")
        print("-" * 65)
        print(reasoning)
        print("-" * 65)
        print(f"  ASR: {asr_time}s | LLM: {llm_time}s | Total: {total_time}s | VRAM: {vram_gb} GB")

    return {
        "audio_path"   : audio_path,
        "transcription": transcription,
        "reasoning"    : reasoning,
        "asr_time_s"   : asr_time,
        "llm_time_s"   : llm_time,
        "total_time_s" : total_time,
        "vram_gb"      : vram_gb,
    }

print("speech_to_reasoning() pipeline defined.")

speech_to_reasoning() pipeline defined.


---
## 🧪 Section 9: Full End-to-End Demonstrations

In [ ]:
# ── Demo 1: Mathematical Reasoning ────────────────────────────
result_1 = speech_to_reasoning(
    "audio_1.mp3",
    query_label    = "Mathematical Reasoning — Train Meeting Problem",
    max_new_tokens = 400,
)

QUERY: Mathematical Reasoning — Train Meeting Problem

[Stage 1 / ASR] Whisper Transcription...


Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Transcription: a train leaves station A at 8 a.m. traveling at 90 kilometers per hour. Another train leaves station B, which is 360 kilometers away, at 9 a.m. traveling at 60 kilometers per hour toward station A. At what time will the two trains meet?
  ASR time: 2.05s

[Stage 2 / LLM] Qwen2.5 Reasoning...
  Prompt tokens: 107


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i


  Reasoned Response:
-----------------------------------------------------------------
To determine when the two trains will meet, we need to calculate how long it takes for them to cover the distance between them given their relative speeds.

1. **Calculate the head start of Train A:**
   - Train A starts at 8 a.m.
   - Train B starts at 9 a.m., so Train A has a 1-hour head start.
   - In that 1 hour, Train A travels \(90 \text{ km/h} \times 1 \text{ h} = 90 \text{ km}\).

2. **Remaining Distance to be Covered:**
   - The initial distance between the two trains was 360 km.
   - After Train A's 1-hour head start, the remaining distance is \(360 \text{ km} - 90 \text{ km} = 270 \text{ km}\).

3. **Combined Speed of Both Trains:**
   - Train A's speed is 90 km/h.
   - Train B's speed is 60 km/h.
   - Their combined speed is \(90 \text{ km/h} + 60 \text{ km/h} = 150 \text{ km/h}\).

4. **Time to Cover Remaining Distance:**
   - To find out how long it takes for both trains to meet after 

In [ ]:
# ── Demo 2: Logical Syllogism ──────────────────────────────────
result_2 = speech_to_reasoning(
    "audio_2.mp3",
    query_label    = "Logical Syllogism — Engineers & Creative Thinkers",
    max_new_tokens = 400,
)

QUERY: Logical Syllogism — Engineers & Creative Thinkers

[Stage 1 / ASR] Whisper Transcription...


Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Transcription: All engineers are problem solvers. Some problem solvers are creative thinkers. Can we conclude that some engineers are creative thinkers? Explain your reasoning step by step.
  ASR time: 1.03s

[Stage 2 / LLM] Qwen2.5 Reasoning...
  Prompt tokens: 77


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



  Reasoned Response:
-----------------------------------------------------------------
To determine if we can conclude that "some engineers are creative thinkers" based on the given statements, let's break down the information provided:

1. **Premise 1:** All engineers are problem solvers.
   - This means every engineer falls under the category of being a problem solver.

2. **Premise 2:** Some problem solvers are creative thinkers.
   - This indicates there exists at least one person who is both an engineer (as they are also a problem solver) and a creative thinker.

Now, let's reason step-by-step to draw the conclusion:

- Since all engineers are problem solvers (Premise 1), and since some problem solvers are creative thinkers (Premise 2), we need to consider what this implies about engineers specifically.

- If some problem solvers are creative thinkers, then there must be at least one individual who fits into both categories: an engineer and a creative thinker.

Therefore, because

In [ ]:
# ── Demo 3: Technical Q&A ──────────────────────────────────────
result_3 = speech_to_reasoning(
    "audio_3.mp3",
    query_label    = "Technical Q&A — Supervised vs Unsupervised ML",
    max_new_tokens = 450,
)

QUERY: Technical Q&A — Supervised vs Unsupervised ML

[Stage 1 / ASR] Whisper Transcription...


Both `max_new_tokens` (=450) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Transcription: What is the difference between supervised learning and unsupervised learning in machine learning? Give a practical example of each.
  ASR time: 0.57s

[Stage 2 / LLM] Qwen2.5 Reasoning...
  Prompt tokens: 68


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



  Reasoned Response:
-----------------------------------------------------------------
To understand the difference between supervised and unsupervised learning, let's first define these terms:

1. **Supervised Learning**: This type of learning involves training a model on labeled data (i.e., data where both input features and output labels are provided). The goal is to learn a mapping from inputs to outputs so that future inputs can be predicted correctly.

2. **Unsupervised Learning**: In contrast, this approach deals with unlabeled data (where only input features are available without corresponding output labels). The objective here is to find hidden patterns or intrinsic structures within the dataset.

### Practical Examples

#### Supervised Learning Example:
**Example: Spam Email Detection**
- **Input Features**: Characteristics like email body text length, presence of certain words (e.g., "free", "win"), sender's IP address, etc.
- **Output Label**: Whether an email is spam (lab

In [ ]:
# ── Demo 4: IE Production Problem ─────────────────────────────
result_4 = speech_to_reasoning(
    "audio_4.mp3",
    query_label    = "IE Problem — Factory Production Calculation",
    max_new_tokens = 350,
)

QUERY: IE Problem — Factory Production Calculation

[Stage 1 / ASR] Whisper Transcription...


Both `max_new_tokens` (=350) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Transcription: In a factory, Machine A produces 200 units per hour and Machine B produces 150 units per hour. Machine A runs for 3 hours. Then breaks down. Machine B continues for 2 more hours after that. What is the total production across both machines?
  ASR time: 1.09s

[Stage 2 / LLM] Qwen2.5 Reasoning...
  Prompt tokens: 103


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



  Reasoned Response:
-----------------------------------------------------------------
To determine the total production of both machines, we need to calculate the production from each machine separately and then sum them up.

### Step 1: Calculate Production from Machine A
- Machine A's production rate = 200 units/hour
- Time Machine A operates = 3 hours

\[ \text{Production by Machine A} = \text{Rate} \times \text{Time} = 200 \, \text{units/hour} \times 3 \, \text{hours} = 600 \, \text{units} \]

### Step 2: Calculate Production from Machine B
- Machine B's production rate = 150 units/hour
- Time Machine B operates = 2 hours (after Machine A stops)

\[ \text{Production by Machine B} = \text{Rate} \times \text{Time} = 150 \, \text{units/hour} \times 2 \, \text{hours} = 300 \, \text{units} \]

### Step 3: Sum Up Total Production
\[ \text{Total Production} = \text{Production by Machine A} + \text{Production by Machine B} = 600 \, \text{units} + 300 \, \text{units} = 900 \, \text{units}

---
## 📦 Section 10: Batch Processing

In [ ]:
import pandas as pd

def batch_speech_to_reasoning(entries: list, max_new_tokens: int = 400) -> list:
    """
    Process a batch of audio queries end-to-end.

    Whisper handles multi-chunk batching internally (batch_size=8).
    LLM queries are sequential with VRAM snapshot after each.

    Args:
        entries        : List of dicts with 'type' and 'path' keys.
        max_new_tokens : Max tokens per LLM response.
    Returns:
        List of result summary dicts.
    """
    results = []
    print(f"Batch processing {len(entries)} audio queries...")
    print("=" * 65)

    for i, entry in enumerate(entries, 1):
        print(f"\n[{i}/{len(entries)}] {entry['type']}")

        t0            = time.time()
        transcription = transcribe_audio(entry["path"], verbose=False)
        asr_time      = round(time.time() - t0, 2)

        t1        = time.time()
        reasoning = generate_reasoning(transcription, max_new_tokens=max_new_tokens,
                                        verbose=False)
        llm_time  = round(time.time() - t1, 2)

        vram_gb = round(torch.cuda.memory_allocated() / 1e9, 2)

        print(f"  Transcription : {transcription[:75]}...")
        print(f"  Response      : {reasoning[:90]}...")
        print(f"  Times: ASR={asr_time}s | LLM={llm_time}s | VRAM={vram_gb} GB")

        results.append({
            "Query Type" : entry["type"],
            "ASR (s)"    : asr_time,
            "LLM (s)"    : llm_time,
            "Total (s)"  : round(asr_time + llm_time, 2),
            "VRAM (GB)"  : vram_gb,
        })

    return results


batch_entries = [{"type": q["type"], "path": f"{q['id']}.mp3"} for q in AUDIO_QUERIES]
batch_results = batch_speech_to_reasoning(batch_entries)

df_batch = pd.DataFrame(batch_results)
print("\nBatch Results:")
print(df_batch.to_string(index=False))

Batch processing 4 audio queries...

[1/4] Mathematical Reasoning


Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


  Transcription : a train leaves station A at 8 a.m. traveling at 90 kilometers per hour. Ano...
  Response      : To determine when the two trains will meet, we can set up an equation based on their relat...
  Times: ASR=1.17s | LLM=21.63s | VRAM=2.63 GB

[2/4] Logical Syllogism


Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


  Transcription : All engineers are problem solvers. Some problem solvers are creative thinke...
  Response      : To determine if we can conclude that "some engineers are creative thinkers" based on the g...
  Times: ASR=0.87s | LLM=14.15s | VRAM=2.63 GB

[3/4] Technical Question Answering


Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


  Transcription : What is the difference between supervised learning and unsupervised learnin...
  Response      : To understand the difference between supervised and unsupervised learning, let's break dow...
  Times: ASR=0.58s | LLM=21.29s | VRAM=2.63 GB

[4/4] Industrial Engineering Problem


Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


  Transcription : In a factory, Machine A produces 200 units per hour and Machine B produces ...
  Response      : To determine the total production from both machines, we need to calculate the production ...
  Times: ASR=1.07s | LLM=13.7s | VRAM=2.63 GB

Batch Results:
                    Query Type  ASR (s)  LLM (s)  Total (s)  VRAM (GB)
        Mathematical Reasoning     1.17    21.63      22.80       2.63
             Logical Syllogism     0.87    14.15      15.02       2.63
  Technical Question Answering     0.58    21.29      21.87       2.63
Industrial Engineering Problem     1.07    13.70      14.77       2.63


---
## 🎙️ Section 11: Upload Your Own Audio File

In [ ]:
from google.colab import files as colab_files

print("Upload an audio file (.mp3 or .wav) — or press Cancel to use the fallback demo.\n")
try:
    uploaded = colab_files.upload()
    for filename, data in uploaded.items():
        with open(filename, 'wb') as f:
            f.write(data)
        print(f"\nUploaded: {filename}")
        result_custom = speech_to_reasoning(filename, query_label=f"Custom: {filename}", max_new_tokens=500)
except Exception:
    print("Upload skipped. Running fallback on audio_1.mp3...")
    result_custom = speech_to_reasoning("audio_1.mp3", query_label="Fallback Demo", max_new_tokens=400)

Upload an audio file (.mp3 or .wav) — or press Cancel to use the fallback demo.



---
## 📊 Section 12: VRAM & Performance Summary

In [ ]:
import pandas as pd

total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
alloc_vram = torch.cuda.memory_allocated() / 1e9
resrv_vram = torch.cuda.memory_reserved()  / 1e9

print("=" * 60)
print("   VRAM USAGE SUMMARY (Speech-to-Reasoning Pipeline)")
print("=" * 60)
print(f"  GPU        : {torch.cuda.get_device_name(0)}")
print(f"  Total VRAM : {total_vram:.2f} GB")
print(f"  Allocated  : {alloc_vram:.2f} GB  ({alloc_vram/total_vram*100:.1f}%)")
print(f"  Reserved   : {resrv_vram:.2f} GB  ({resrv_vram/total_vram*100:.1f}%)")
print(f"  Free       : {total_vram-resrv_vram:.2f} GB  ({(total_vram-resrv_vram)/total_vram*100:.1f}%)")
print("=" * 60)
print("\n  VRAM breakdown:")
print("    Whisper-small fp16              : ~0.5 GB")
print("    Qwen2.5-3B dynamic 4-bit        : ~2.0 GB")
print("    KV cache + activations          : ~0.3 GB")
print("    Combined peak                   : ~2.8 GB")
print("    Headroom remaining on T4        : ~12.8 GB")
print("=" * 60)

# Per-query table
all_results = [result_1, result_2, result_3, result_4]
labels = ["Math Reasoning", "Logical Syllogism", "Technical Q&A", "IE Production"]
rows = [{"Query": lbl, "ASR (s)": r["asr_time_s"],
         "LLM (s)": r["llm_time_s"], "Total (s)": r["total_time_s"],
         "VRAM (GB)": r["vram_gb"]} for r, lbl in zip(all_results, labels)]
df = pd.DataFrame(rows)
print("\nPer-query Performance:")
print(df.to_string(index=False))
avg_asr = df["ASR (s)"].mean()
avg_llm = df["LLM (s)"].mean()
print(f"\n  Avg ASR: {avg_asr:.2f}s | Avg LLM: {avg_llm:.2f}s | Avg Total: {avg_asr+avg_llm:.2f}s")

   VRAM USAGE SUMMARY (Speech-to-Reasoning Pipeline)
  GPU        : Tesla T4
  Total VRAM : 15.64 GB
  Allocated  : 2.63 GB  (16.8%)
  Reserved   : 3.06 GB  (19.5%)
  Free       : 12.58 GB  (80.5%)

  VRAM breakdown:
    Whisper-small fp16              : ~0.5 GB
    Qwen2.5-3B dynamic 4-bit        : ~2.0 GB
    KV cache + activations          : ~0.3 GB
    Combined peak                   : ~2.8 GB
    Headroom remaining on T4        : ~12.8 GB

Per-query Performance:
            Query  ASR (s)  LLM (s)  Total (s)  VRAM (GB)
   Math Reasoning     2.05    36.27      38.32       2.63
Logical Syllogism     1.03    12.89      13.93       2.63
    Technical Q&A     0.57    21.92      22.49       2.63
    IE Production     1.09    23.48      24.57       2.63

  Avg ASR: 1.19s | Avg LLM: 23.64s | Avg Total: 24.83s


---
## ✅ Section 13: Automated Verification Checklist

In [ ]:
print("=" * 60)
print("    SPEECH-TO-REASONING PIPELINE — VERIFICATION")
print("=" * 60)

checks = [
    ("Whisper ASR loaded (fp16, GPU)",              asr_pipe is not None),
    ("Qwen2.5-3B dynamic 4-bit LLM loaded",         llm_model is not None),
    ("4 audio files generated via gTTS",            all(os.path.exists(f"audio_{i}.mp3") for i in range(1, 5))),
    ("All transcriptions non-empty",                all(len(r["transcription"]) > 10 for r in all_results)),
    ("All reasoning responses non-empty",           all(len(r["reasoning"]) > 20 for r in all_results)),
    ("VRAM under 5 GB (quantization effective)",    torch.cuda.memory_allocated() / 1e9 < 5.0),
    ("4 queries completed end-to-end",              len(all_results) == 4),
    ("Batch function executed successfully",        len(batch_results) == 4),
]

all_pass = True
for label, passed in checks:
    status = "PASS" if passed else "FAIL"
    symbol = "OK" if passed else "XX"
    all_pass = all_pass and passed
    print(f"  [{symbol}] {status:<6} {label}")

print("=" * 60)
if all_pass:
    print("\n  ALL CHECKS PASSED — Task 4 Complete!")
else:
    print("\n  Some checks failed — review output above.")
print("=" * 60)

    SPEECH-TO-REASONING PIPELINE — VERIFICATION
  [OK] PASS   Whisper ASR loaded (fp16, GPU)
  [OK] PASS   Qwen2.5-3B dynamic 4-bit LLM loaded
  [OK] PASS   4 audio files generated via gTTS
  [OK] PASS   All transcriptions non-empty
  [OK] PASS   All reasoning responses non-empty
  [OK] PASS   VRAM under 5 GB (quantization effective)
  [OK] PASS   4 queries completed end-to-end
  [OK] PASS   Batch function executed successfully

  ALL CHECKS PASSED — Task 4 Complete!


---
## 📋 Summary

| Component | Choice | Detail |
|---|---|---|
| **ASR Model** | `openai/whisper-small` | fp16, batch_size=8, 30s window, stride overlap |
| **Reasoning LLM** | `Qwen2.5-3B-Instruct-bnb-4bit` | Unsloth dynamic 4-bit, ~2 GB VRAM |
| **Audio Input** | gTTS + user upload | Math, Logic, QA, IE problem types |
| **Encoding** | fp16 (Whisper) + int4/fp16 mix (LLM) | Both run on T4 GPU |
| **Batching** | batch_size=8 audio chunks (Whisper) | Sequential LLM with VRAM tracking |
| **Peak VRAM** | ~2.8 GB | <18% of T4 budget |

**End-to-end flow:**
```
Audio File (.mp3 / .wav)
      |
      v
[Whisper-small  fp16  batch_size=8]
      |  Transcribed text
      v
[build_prompt()  Qwen2.5 ChatML format]
      |
      v
[Qwen2.5-3B-Instruct  dynamic 4-bit]
      |  Reasoned step-by-step response
      v
Result dict: transcription, reasoning, timings, VRAM
```